In [ ]:
from dotenv import load_dotenv
load_dotenv()

In [ ]:
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_chroma import Chroma
from langchain_groq import ChatGroq
from langchain_core.prompts import PromptTemplate






### Data Loading


In [ ]:
loader = PyPDFLoader("../data/data_science_syllabus.pdf")
docs = loader.load()
len(docs)

### Text Splitting

In [ ]:
splitter = RecursiveCharacterTextSplitter(chunk_size=1000,chunk_overlap=200)
splitted_data = splitter.split_documents(docs)
len(splitted_data)

### Creating Embeddings

In [ ]:
embeddings = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2"
)

In [ ]:
vector_store= Chroma.from_documents(
    documents= splitted_data,
    embedding=embeddings
)

In [ ]:
query = "Machine Learnings and Data Science Content"
data = vector_store.similarity_search(query=query)
len(data)

In [ ]:
context =""
for doc in data:
    context+= doc.page_content+("\n")

print(context)    

In [ ]:
llm = ChatGroq(
    model="openai/gpt-oss-20b"
)

### Creating chain - Context generation | prompt | llm | strparser

In [ ]:
def get_context(query:str):
    data = vector_store.similarity_search(query=query)
    context =""
    for doc in data:
        context+= doc.page_content+("\n")

    return {
    "context":context,
    "question":query
    }
    

In [ ]:
prompt = PromptTemplate.from_template("""
    You are a helpful assistant and provide answerd based on the context for user question. and 
    if you don't know the answer, then you can say that 'I dont know.'
    Context: {context}
    Question: {question}
""")
                        

In [ ]:
rag_chain = get_context | prompt | llm

In [ ]:
res = rag_chain.invoke("Give me complete content for gen Ai module")

In [ ]:
print(res.content)